# AMPI Demo — Approximate Nearest Neighbors on MNIST

**Dataset**: MNIST (70 000 images, d = 784)

**Methods compared**:

| Method | Type | Library |
|---|---|---|
| FAISS FlatL2 | Exact brute force | FAISS |
| PyKeOps | Exact brute force (lazy) | PyKeOps |
| FAISS IVF | Approximate (inverted file) | FAISS |
| AMPI Binary | Approximate (sorted projections) | this repo |
| AMPI AffineFan | Approximate (k-means + fan cones + sorted projections) | this repo |

We measure **Recall\@k** (fraction of true top-k neighbours returned) and **ms/query**.

See `README.md` for the mathematical background.

In [ ]:
import sys, os, time, struct, tempfile
import numpy as np
import faiss

import ampi
from ampi import AMPIBinaryIndex, AMPIAffineFanIndex, AFanTuner
from ampi import save_checkpoint, load_checkpoint
from ampi import WALWriter, replay_wal, truncate_wal

print(f'ampi version: {ampi.__version__}')

try:
    import torch
    from pykeops.torch import LazyTensor
    HAS_KEOPS = True
    print('PyKeOps available')
except ImportError:
    HAS_KEOPS = False
    print('PyKeOps not installed — skipping (pip install pykeops torch)')

SEED = 42
K    = 10      # neighbours
NQ   = 500     # number of query points
print(f'faiss version: {faiss.__version__}')

# Shared temp dir for streaming build / WAL / checkpoint demos
_TMPDIR = tempfile.mkdtemp(prefix='ampi_demo_')
print(f'temp dir: {_TMPDIR}')

In [ ]:
# ── Load MNIST from the raw IDX files (no torchvision required) ──────────────

def _read_idx_images(path):
    with open(path, 'rb') as f:
        magic, n, rows, cols = struct.unpack('>IIII', f.read(16))
        assert magic == 2051
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(n, rows * cols)

def _read_idx_labels(path):
    with open(path, 'rb') as f:
        magic, n = struct.unpack('>II', f.read(8))
        assert magic == 2049
        return np.frombuffer(f.read(), dtype=np.uint8)

root = 'data/MNIST/raw'
X_train = _read_idx_images(f'{root}/train-images-idx3-ubyte')
X_test  = _read_idx_images(f'{root}/t10k-images-idx3-ubyte')
y_train = _read_idx_labels(f'{root}/train-labels-idx1-ubyte')
y_test  = _read_idx_labels(f'{root}/t10k-labels-idx1-ubyte')

# Stack and normalise to [0, 1] float32
X = np.vstack([X_train, X_test]).astype(np.float32) / 255.0
y = np.concatenate([y_train, y_test])
n, d = X.shape

rng     = np.random.default_rng(SEED)
q_idx   = rng.choice(n, NQ, replace=False)
queries = X[q_idx]   # shape (NQ, 784)

print(f'Dataset: n={n:,}  d={d}  queries={NQ}')

In [ ]:
# ── Ground truth: FAISS FlatL2 (exact brute force) ───────────────────────────

flat = faiss.IndexFlatL2(d)
flat.add(X)

t0 = time.perf_counter()
_, I_true = flat.search(queries, K)
flat_total_ms = (time.perf_counter() - t0) * 1000

print(f'FAISS FlatL2  total {flat_total_ms:.0f} ms  ({flat_total_ms/NQ:.2f} ms/query)')

In [ ]:
# ── PyKeOps brute force (lazy, no O(n²) materialisation) ─────────────────────

if HAS_KEOPS:
    X_t  = torch.from_numpy(X)
    Q_t  = torch.from_numpy(queries)
    X_i  = LazyTensor(Q_t[:, None, :])   # (NQ, 1, d)
    X_j  = LazyTensor(X_t[None, :, :])   # (1, n, d)
    D_ij = ((X_i - X_j) ** 2).sum(-1)   # (NQ, n)

    t0 = time.perf_counter()
    I_keops = D_ij.argKmin(K, dim=1).numpy()   # (NQ, K)
    keops_total_ms = (time.perf_counter() - t0) * 1000

    # Verify it matches FAISS
    overlap = np.mean([
        len(set(I_true[i]) & set(I_keops[i])) / K
        for i in range(NQ)
    ])
    print(f'PyKeOps       total {keops_total_ms:.0f} ms  ({keops_total_ms/NQ:.2f} ms/query)  '
          f'recall@{K}={overlap:.4f} (vs FAISS Flat)')
else:
    print('PyKeOps skipped.')

In [ ]:
# ── Build AMPI and FAISS IVF indices ─────────────────────────────────────────

print('Building AMPI Binary     (L=16)…', end=' ', flush=True)
t0 = time.perf_counter()
idx_bin = AMPIBinaryIndex(X, num_projections=16, seed=SEED)
print(f'{time.perf_counter()-t0:.2f}s')

print('Building AMPI AffineFan  (F=16, nlist=auto)…', end=' ', flush=True)
t0 = time.perf_counter()
idx_fan = AMPIAffineFanIndex(X, num_fans=16, seed=SEED)
print(f'{time.perf_counter()-t0:.2f}s')

nlist = max(64, int(n * 0.01))
print(f'Building FAISS IVF       (nlist={nlist})…', end=' ', flush=True)
t0 = time.perf_counter()
quant = faiss.IndexFlatL2(d)
ivf   = faiss.IndexIVFFlat(quant, d, nlist, faiss.METRIC_L2)
ivf.train(X); ivf.add(X)
print(f'{time.perf_counter()-t0:.2f}s')

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def recall_at_k(I_true, I_approx, k):
    hits = sum(len(set(t[:k]) & set(a[:k])) for t, a in zip(I_true, I_approx))
    return hits / (len(I_true) * k)

def warmup_time(fn, items, warmup=5):
    for x in items[:warmup]: fn(x)
    t0 = time.perf_counter()
    for x in items:          fn(x)
    return (time.perf_counter() - t0) / len(items) * 1000

def avg_cands(idx, queries, **kw):
    return int(np.mean([len(idx.query_candidates(q, **kw)) for q in queries[:20]]))

def run_config(label, query_fn, cands_fn=None):
    rows = [query_fn(q) for q in queries]
    I    = np.array([np.pad(r, (0, max(0, K - len(r))), constant_values=-1) for r in rows])
    ms   = warmup_time(query_fn, queries)
    cstr = f'{cands_fn():>8,}' if cands_fn else f'{"—":>8}'
    r1   = recall_at_k(I_true, I, 1)
    r5   = recall_at_k(I_true, I, 5)
    rk   = recall_at_k(I_true, I, K)
    print(f'  {label:<26}  {ms:>8.2f}  {cstr}  {r1:>6.3f}  {r5:>6.3f}  {rk:>6.3f}')

In [ ]:
# ── Summary comparison ────────────────────────────────────────────────────────

print(f'  {"Method":<30}  {"ms/q":>8}  {"cands":>8}  {"R@1":>6}  {"R@5":>6}  {f"R@{K}":>6}')
print('  ' + '-' * 76)

# FAISS Flat (exact)
ms = warmup_time(lambda q: flat.search(q[None], K), queries)
print(f'  {"Flat L2 (exact)":<30}  {ms:>8.2f}  {n:>8,}  {1.0:>6.3f}  {1.0:>6.3f}  {1.0:>6.3f}')

# PyKeOps
if HAS_KEOPS:
    ms_keops = keops_total_ms / NQ
    print(f'  {"PyKeOps (exact)":<30}  {ms_keops:>8.2f}  {n:>8,}  {1.0:>6.3f}  {1.0:>6.3f}  {1.0:>6.3f}')

# FAISS IVF
for nprobe in [1, 5, 20]:
    ivf.nprobe = nprobe
    rows = [ivf.search(q[None], K)[1][0] for q in queries]
    I_ivf = np.array(rows)
    ms  = warmup_time(lambda q: ivf.search(q[None], K), queries)
    r1  = recall_at_k(I_true, I_ivf, 1)
    r5  = recall_at_k(I_true, I_ivf, 5)
    rk  = recall_at_k(I_true, I_ivf, K)
    cds = nprobe * (n // nlist)
    print(f'  {f"IVF nprobe={nprobe}":<30}  {ms:>8.2f}  {cds:>8,}  {r1:>6.3f}  {r5:>6.3f}  {rk:>6.3f}')

# AMPI Binary
for w in [25, 100, 400]:
    run_config(
        f'AMPI Binary  w={w}',
        lambda q, w=w: idx_bin.query(q, k=K, window_size=w)[2],
        lambda w=w: avg_cands(idx_bin, queries, window_size=w),
    )

# AMPI AffineFan
for cp, fp, w in [(5, 2, 50), (10, 4, 50), (20, 8, 100)]:
    run_config(
        f'AMPI AffineFan cp={cp} fp={fp} w={w}',
        lambda q, cp=cp, fp=fp, w=w: idx_fan.query(q, k=K, window_size=w, probes=cp, fan_probes=fp)[2],
        lambda cp=cp, fp=fp, w=w: avg_cands(idx_fan, queries, window_size=w, probes=cp, fan_probes=fp),
    )

## AffineFan parameter sweep: `probes` × `fan_probes`

`probes` (cp) controls how many k-means clusters are visited; `fan_probes` (fp) controls how many fan cones per cluster.  
Increasing either raises recall at the cost of more candidates.

In [ ]:
print(f'  AMPI AffineFan (F=16, w=50 fixed) — vary probes × fan_probes')
print(f'  {"probes":>8}  {"fan_probes":>10}  {"cands":>8}  {"ms/q":>8}  {"R@1":>6}  {"R@5":>6}  {f"R@{K}":>6}')
print('  ' + '-' * 72)

for cp in [3, 5, 10, 20]:
    for fp in [1, 2, 4, 8]:
        rows = [idx_fan.query(q, k=K, window_size=50, probes=cp, fan_probes=fp)[2] for q in queries]
        I    = np.array([np.pad(r, (0, max(0, K - len(r))), constant_values=-1) for r in rows])
        ms   = warmup_time(lambda q, cp=cp, fp=fp: idx_fan.query(q, k=K, window_size=50, probes=cp, fan_probes=fp), queries)
        cds  = int(np.mean([len(idx_fan.query_candidates(q, window_size=50, probes=cp, fan_probes=fp)) for q in queries[:20]]))
        r1   = recall_at_k(I_true, I, 1)
        r5   = recall_at_k(I_true, I, 5)
        rk   = recall_at_k(I_true, I, K)
        print(f'  {cp:>8}  {fp:>10}  {cds:>8,}  {ms:>8.2f}  {r1:>6.3f}  {r5:>6.3f}  {rk:>6.3f}')

## Binary vs AffineFan: iso-recall curve

For a fixed recall target, which method needs fewer candidates (= faster re-ranking)?  
Binary is adaptive in density; AffineFan adds geometric structure via k-means partitioning.

In [ ]:
print(f'  Binary vs AffineFan — candidates vs recall@{K}')
print(f'  {"Method":<28}  {"cands":>8}  {"ms/q":>8}  {f"R@{K}":>6}')
print('  ' + '-' * 58)

# AMPI Binary
for w in [10, 25, 50, 100, 200, 400, 800]:
    fn = lambda q, w=w: idx_bin.query(q, k=K, window_size=w)[2]
    cf = lambda w=w: int(np.mean([len(idx_bin.query_candidates(q, window_size=w)) for q in queries[:20]]))
    rows  = [fn(q) for q in queries]
    I     = np.array([np.pad(r, (0, max(0, K - len(r))), constant_values=-1) for r in rows])
    ms    = warmup_time(fn, queries)
    cands = cf()
    rk    = recall_at_k(I_true, I, K)
    print(f'  {f"Binary w={w}":<28}  {cands:>8,}  {ms:>8.2f}  {rk:>6.3f}')

print()

# AMPI AffineFan — vary probes with fp=4, w=50
for cp in [1, 2, 3, 5, 10, 20, 40]:
    fn = lambda q, cp=cp: idx_fan.query(q, k=K, window_size=50, probes=cp, fan_probes=4)[2]
    cf = lambda cp=cp: int(np.mean([len(idx_fan.query_candidates(q, window_size=50, probes=cp, fan_probes=4)) for q in queries[:20]]))
    rows  = [fn(q) for q in queries]
    I     = np.array([np.pad(r, (0, max(0, K - len(r))), constant_values=-1) for r in rows])
    ms    = warmup_time(fn, queries)
    cands = cf()
    rk    = recall_at_k(I_true, I, K)
    print(f'  {f"AffineFan cp={cp} fp=4 w=50":<28}  {cands:>8,}  {ms:>8.2f}  {rk:>6.3f}')

## AFanTuner: automatic build- and query-parameter selection

`AFanTuner` runs Bayesian optimisation on a data sample to find the best `nlist` (alpha · √n) and `cone_top_k`, then sweeps query parameters to suggest (cp, fp, w) configs for each target recall level.

In [ ]:
# ── AFanTuner demo ────────────────────────────────────────────────────────────
# Uses NQ queries + I_true as ground truth.

tuner  = AFanTuner(X, queries, I_true, n_bo_iter=8, k=K, seed=SEED)
result = tuner.tune(verbose=True)

idx_tuned = result['index']
print(f"\nTuned index: nlist={result['nlist']}  alpha={result['alpha']:.3f}"
      f"  cone_top_k={result['K']}  num_fans={result['F']}")

In [ ]:
# ── Tuner suggestions: cheapest (cp, fp, w) for each recall target ────────────
print(f'  {"Target R@K":>10}  {"cp":>4}  {"fp":>4}  {"w":>5}  {"cands":>8}  {"Actual R@K":>10}')
print('  ' + '-' * 54)
for target_recall, cp, fp, w, cands, actual_recall in result['suggestions']:
    print(f'  {target_recall:>10.2f}  {cp:>4}  {fp:>4}  {w:>5}  {cands:>8,}  {actual_recall:>10.4f}')

In [ ]:
# ── Empirical NN distance stats (helps calibrate bucket_size) ────────────────
# For each query, compute the distance to its 10th nearest neighbour.

nn_dists = np.sqrt(np.array([
    flat.search(q[None], K)[0][0, K-1] for q in queries[:200]
]))

sigma_proj = nn_dists / np.sqrt(d)

print(f'MNIST nearest-neighbour distances (L2, k={K})')
print(f'  r_nn  :  mean={nn_dists.mean():.2f}  median={np.median(nn_dists):.2f}  p10={np.percentile(nn_dists,10):.2f}  p90={np.percentile(nn_dists,90):.2f}')
print(f'  σ_proj = r_nn/√d:  mean={sigma_proj.mean():.3f}  median={np.median(sigma_proj):.3f}')
print(f'  Recommended bucket_size ≈ {np.median(sigma_proj):.2f}')

## Streaming mutations: add / delete / batch_add / update

`AMPIAffineFanIndex` supports mutable operations without a full rebuild.

| Method | Description |
|---|---|
| `add(x)` | Insert one vector; returns its `global_id` |
| `delete(global_id)` | Logical tombstone; auto-compacts when the per-cluster tombstone fraction exceeds a threshold |
| `update(global_id, x)` | Atomic delete + insert; returns new `global_id` |
| `batch_add(data)` | Bulk insert `(m, d)` array; returns `(m,)` `global_ids` |
| `batch_delete(global_ids)` | Bulk tombstone |

Recall is measured before and after mutations to verify correctness.

In [ ]:
# ── Streaming mutations demo ──────────────────────────────────────────────────
# Build a small index on a subset so mutations are fast in the notebook.
rng2   = np.random.default_rng(SEED + 1)
n_demo = 5_000
demo_idx = rng2.choice(n, n_demo, replace=False)
X_demo   = X[demo_idx]

idx_mut = AMPIAffineFanIndex(X_demo, seed=SEED)

# Ground truth for demo queries
n_dq    = 50
dq_idx  = rng2.choice(n_demo, n_dq, replace=False)
dq      = X_demo[dq_idx]
flat_d  = faiss.IndexFlatL2(d)
flat_d.add(X_demo)
_, I_demo_gt = flat_d.search(dq, K)

def demo_recall(idx, qs, gt):
    rows = [idx.query(q, k=K, window_size=100, probes=5, fan_probes=2)[2] for q in qs]
    return recall_at_k(gt, np.array([np.pad(r, (0, max(0, K-len(r))), constant_values=-1) for r in rows]), K)

print(f'Baseline recall@{K}: {demo_recall(idx_mut, dq, I_demo_gt):.4f}  (n={idx_mut.n:,})')

# ── add() ────────────────────────────────────────────────────────────────────
new_vec = rng2.standard_normal(d).astype(np.float32)
new_id  = idx_mut.add(new_vec)
print(f'add()        → global_id={new_id}  n={idx_mut.n:,}')

# ── delete() ─────────────────────────────────────────────────────────────────
idx_mut.delete(new_id)
print(f'delete()     → n={idx_mut.n:,} (tombstone; compacts automatically)')

# ── batch_add() ──────────────────────────────────────────────────────────────
extra = rng2.standard_normal((200, d)).astype(np.float32)
new_ids = idx_mut.batch_add(extra)
print(f'batch_add()  → global_ids {new_ids[0]}…{new_ids[-1]}  n={idx_mut.n:,}')

# ── batch_delete() ───────────────────────────────────────────────────────────
idx_mut.batch_delete(new_ids)
print(f'batch_delete()→ n={idx_mut.n:,}')

# ── update() ─────────────────────────────────────────────────────────────────
target_id  = 0
updated_id = idx_mut.update(target_id, rng2.standard_normal(d).astype(np.float32))
print(f'update()     → new global_id={updated_id}')

# Recall after mutations (should be close to baseline)
print(f'Post-mutation recall@{K}: {demo_recall(idx_mut, dq, I_demo_gt):.4f}')

## Metric support: l2 / sqeuclidean / cosine

`AMPIAffineFanIndex` accepts a `metric` argument at build time.  
For `cosine`, vectors are L2-normalised internally and distances are `1 − cos_sim`.

In [ ]:
# ── Metric comparison on demo subset ─────────────────────────────────────────
# Cosine similarity is natural for unit-normed data; MNIST is not unit-normed,
# so we normalise before building the cosine index for a fair comparison.

X_norm = (X_demo / np.linalg.norm(X_demo, axis=1, keepdims=True).clip(1e-10)).astype(np.float32)
dq_norm = (dq      / np.linalg.norm(dq,      axis=1, keepdims=True).clip(1e-10)).astype(np.float32)

# Cosine ground truth via FAISS (inner-product on unit vectors = cosine)
flat_cos = faiss.IndexFlatIP(d)
flat_cos.add(X_norm)
_, I_cos_gt = flat_cos.search(dq_norm, K)

Q_ARGS = dict(k=K, window_size=100, probes=5, fan_probes=2)

print(f'  {"Metric":<14}  {"R@K":>6}  {"dist example":>14}')
print('  ' + '-' * 40)

for metric, data_in, dq_in, gt_in in [
    ('l2',          X_demo, dq,      I_demo_gt),
    ('sqeuclidean', X_demo, dq,      I_demo_gt),
    ('cosine',      X_norm, dq_norm, I_cos_gt),
]:
    idx_m = AMPIAffineFanIndex(data_in, seed=SEED, metric=metric)
    rows  = [idx_m.query(q, **Q_ARGS) for q in dq_in]
    I_m   = np.array([np.pad(r[2], (0, max(0, K-len(r[2]))), constant_values=-1) for r in rows])
    dist0 = rows[0][1][0]   # distance to 1st neighbour of first query
    rk    = recall_at_k(gt_in, I_m, K)
    print(f'  {metric:<14}  {rk:>6.4f}  {dist0:>14.5f}')

## Streaming build: low-RSS construction via `streaming_build`

`streaming_build` makes a single sequential pass over the data using a callable `data_source(start, end) → array`.
It never loads the full dataset into RAM at once, keeping peak RSS ≈ 90 MB even for GIST-1M.
When `data_path=` is supplied the raw vectors are mmap'd to disk instead of kept in heap memory — required for checkpoint/reload.

| Parameter | Meaning |
|---|---|
| `data_source` | `callable(s, e) → float32[e-s, d]` — called in order |
| `n`, `d` | total vectors and dimensionality |
| `nlist` | number of k-means clusters |
| `data_path` | directory for the mmap `_cpp_data_buf.dat` file |

In [ ]:
# ── Streaming build demo ──────────────────────────────────────────────────────
# Uses the same 5 000-point demo subset; data_path= writes the mmap backing
# file so this index can later be checkpointed and reloaded.
from ampi.streaming import streaming_build

stream_data_path = os.path.join(_TMPDIR, 'stream_data')
os.makedirs(stream_data_path, exist_ok=True)

print(f'streaming_build n={n_demo:,}  d={d}  nlist=32 …', end=' ', flush=True)
t0 = time.perf_counter()
idx_stream = streaming_build(
    data_source=lambda s, e: X_demo[s:e],
    n=n_demo, d=d,
    nlist=32,
    num_fans=16,
    seed=SEED,
    data_path=stream_data_path,
)
print(f'{time.perf_counter()-t0:.2f}s')

mmap_bytes = os.path.getsize(os.path.join(stream_data_path, '_cpp_data_buf.dat'))
print(f'mmap file size: {mmap_bytes:,} bytes  ({mmap_bytes/1024:.1f} kB)')

# Verify recall matches a normally-built index
rows_s = [idx_stream.query(q, k=K, window_size=100, probes=5, fan_probes=2)[2] for q in dq]
I_stream = np.array([np.pad(r, (0, max(0, K-len(r))), constant_values=-1) for r in rows_s])
rk_stream = recall_at_k(I_demo_gt, I_stream, K)

rows_n = [idx_mut.query(q, k=K, window_size=100, probes=5, fan_probes=2)[2] for q in dq]
I_normal = np.array([np.pad(r, (0, max(0, K-len(r))), constant_values=-1) for r in rows_n])
rk_normal = recall_at_k(I_demo_gt, I_normal, K)

print(f'Recall@{K}  streaming_build={rk_stream:.4f}   normal build={rk_normal:.4f}')

## WAL (Write-Ahead Log): durable mutation logging

Pass `wal_path=` to `AMPIAffineFanIndex` (or `streaming_build`) to enable automatic WAL logging.
Every `add()` and `delete()` is appended to the file as a CRC32-protected record.

On restart, `replay_wal(idx, path, after_timestamp_ns=ts)` re-applies mutations since the last checkpoint.
`truncate_wal(path, d)` resets the file to a header-only state after a successful checkpoint.

Wire format per record: `op(1B) | global_id(8B) | has_vector(1B) | vector(d×4B) | timestamp_ns(8B) | CRC32(4B)`

In [ ]:
# ── WAL demo ──────────────────────────────────────────────────────────────────
wal_path = os.path.join(_TMPDIR, 'mutations.wal')

# Build with WAL enabled: every add/delete is logged automatically.
idx_wal = AMPIAffineFanIndex(X_demo, seed=SEED, wal_path=wal_path)
print(f'WAL file created: {os.path.getsize(wal_path):,} bytes (header only)')

# Perform mutations — all logged to WAL
rng3      = np.random.default_rng(SEED + 2)
extra50   = rng3.standard_normal((50, d)).astype(np.float32)
logged_ids = idx_wal.batch_add(extra50)
idx_wal.batch_delete(logged_ids[:10])
print(f'After 50 inserts + 10 deletes: WAL = {os.path.getsize(wal_path):,} bytes')

# Simulate a restart: build a fresh in-memory index, replay the WAL
idx_fresh = AMPIAffineFanIndex(X_demo, seed=SEED)
n_applied = replay_wal(idx_fresh, wal_path)
print(f'Replayed {n_applied} records → n={idx_fresh.n:,}')

# Recall should match the live index
r_live  = demo_recall(idx_wal,   dq, I_demo_gt)
r_fresh = demo_recall(idx_fresh, dq, I_demo_gt)
print(f'Recall@{K}  live={r_live:.4f}  after replay={r_fresh:.4f}')

# Truncate WAL once a checkpoint is taken (see next section)
truncate_wal(wal_path, d)
print(f'WAL truncated → {os.path.getsize(wal_path):,} bytes (header only)')

## Checkpoint: save and reload the full index

`save_checkpoint(idx, path)` serializes centroids, axes, and all sorted cone arrays to a compact binary file.
It returns a `timestamp_ns` you pass to `replay_wal` to skip mutations already captured in the checkpoint.

`load_checkpoint(path, data_path=)` deserializes the file and returns a ready-to-query index.
The raw vectors are **not** stored in the checkpoint — they stay in the mmap file at `data_path/_cpp_data_buf.dat`,
which must be preserved alongside the checkpoint.

Typical shutdown / startup sequence:

```
# shutdown
ts = save_checkpoint(idx, 'index.ckpt')
truncate_wal('mutations.wal', d)

# startup
idx = load_checkpoint('index.ckpt', data_path='/path/to/data')
replay_wal(idx, 'mutations.wal', after_timestamp_ns=ts)
```

In [ ]:
# ── Checkpoint demo ───────────────────────────────────────────────────────────
# Uses idx_stream (built above with data_path=stream_data_path), which has the
# mmap backing file needed for load_checkpoint.
ckpt_path  = os.path.join(_TMPDIR, 'index.ckpt')
wal2_path  = os.path.join(_TMPDIR, 'mutations2.wal')

# Attach a WAL so post-checkpoint mutations are durable
idx_stream._wal = WALWriter(wal2_path, idx_stream.d)

# ── Checkpoint ────────────────────────────────────────────────────────────────
ts = save_checkpoint(idx_stream, ckpt_path)
print(f'Checkpoint saved → {os.path.getsize(ckpt_path):,} bytes  ts={ts}')

# ── Post-checkpoint mutations ─────────────────────────────────────────────────
rng4     = np.random.default_rng(SEED + 3)
post_vecs = rng4.standard_normal((30, d)).astype(np.float32)
post_ids  = idx_stream.batch_add(post_vecs)
idx_stream.batch_delete(post_ids[:5])
print(f'Post-checkpoint: 30 inserts + 5 deletes logged to WAL ({os.path.getsize(wal2_path):,} bytes)')

r_before = demo_recall(idx_stream, dq, I_demo_gt)
print(f'Recall@{K} live (n={idx_stream.n:,}): {r_before:.4f}')

# ── Simulate restart: load checkpoint + replay post-checkpoint WAL ────────────
idx_restored = load_checkpoint(ckpt_path, data_path=stream_data_path)
n_replayed   = replay_wal(idx_restored, wal2_path, after_timestamp_ns=ts)
print(f'Restored: n={idx_restored.n:,}  WAL records replayed: {n_replayed}')

r_after = demo_recall(idx_restored, dq, I_demo_gt)
print(f'Recall@{K} after restore: {r_after:.4f}  (should match live)')

# Truncate WAL — checkpoint is now stable
truncate_wal(wal2_path, idx_restored.d)
print(f'WAL truncated → {os.path.getsize(wal2_path):,} bytes')